# Leakage-safe learned transfer entropy

Estimates transfer entropy (TE) from X to Y as the predictive gain
T̂ = ½ ln(MSE without X / MSE with X), using a pipeline built to avoid leakage (time splits with
gaps, preprocessing fitted on train only, a held-out test block used once), plus diagnostics that
flag spurious transfer from noise. Code and details: [Transfer-entropy-](https://github.com/Gaire-commits/Transfer-entropy-).

**Sections**
1. Setup
2. Simulated systems with known answers
3. Market data from Alpaca: crypto (no keys needed), daily stocks, intraday stocks
4. Your own CSV

Run the cells top to bottom. TE values are in nats.

## 1. Setup

In [ ]:
import os, sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO = "/content/Transfer-entropy-"
    if not os.path.exists(REPO):
        !git clone -q https://github.com/Gaire-commits/Transfer-entropy-.git {REPO}
    else:
        !git -C {REPO} pull -q
    !pip install -q alpaca-py
    os.chdir(REPO)
else:  # running locally from a clone
    REPO = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
sys.path.insert(0, REPO)

# Forget any learned_te code loaded earlier in this session, so re-running this
# cell after the pull above always uses the latest version (no restart needed).
for name in [m for m in sys.modules if m == "learned_te" or m.startswith("learned_te.")]:
    del sys.modules[name]

import warnings
warnings.filterwarnings("ignore")  # library noise (sklearn, websockets)
warnings.filterwarnings("default", module="learned_te")  # but show e.g. skipped short days
import pandas as pd
from learned_te import Config, report, run, simulate, SCENARIOS
import learned_te.split as _split
print("ready:", REPO, "| half-day fix loaded:", hasattr(_split, "MIN_BLOCK"))

In [ ]:
def summarize(rep, label):
    """One row per analysis: forward estimate, reverse estimate, verdict."""
    f, r = rep["forward"], rep["reverse"]
    return {
        "analysis": label,
        "T_eff X->Y": round(f.te_eff, 5),
        "95% CI": f"[{f.ci[0]:+.5f}, {f.ci[1]:+.5f}]",
        "p": round(f.p_value, 3),
        "T_eff Y->X": round(r.te_eff, 5),
        "verdict": rep["verdict"],
    }

results = []  # every section appends here; shown together at the end

## 2. Simulated systems with known answers

Only `coupled` has a real X→Y link. `proxy_leak` and `shared_noise` are the two ways noise
creates spurious TE; the diagnostics should flag them even though they pass the significance test.

In [ ]:
cfg = Config(n_surrogates=100, n_bootstrap=300)
for name, (truth, desc) in SCENARIOS.items():
    rep = report(simulate(name, seed=1), cfg)
    results.append(summarize(rep, f"sim: {name} ({'coupled' if truth else 'no link'})"))
pd.DataFrame(results).set_index("analysis")

## 3. Market data from Alpaca

Returns, not prices, go into the estimator. For two stocks, most co-movement is the market
itself (shared noise), so SPY is passed as a reference column and regressed out.

**Keys** are only needed for stocks. In Colab, click the key icon in the left sidebar and add
two secrets, `APCA_API_KEY_ID` and `APCA_API_SECRET_KEY`, with notebook access on. Free
paper-trading keys work. Don't paste keys into a cell.

In [ ]:
from learned_te.market import fetch_bars, log_returns, to_array, intraday_sessions, last_sessions

def have_stock_keys():
    try:
        from google.colab import userdata
        return bool(userdata.get("APCA_API_KEY_ID") and userdata.get("APCA_API_SECRET_KEY"))
    except Exception:
        return bool(os.environ.get("APCA_API_KEY_ID") and os.environ.get("APCA_API_SECRET_KEY"))

print("stock keys found:", have_stock_keys())

### 3a. Crypto, hourly (no keys needed)

In [ ]:
prices = fetch_bars(["BTC/USD", "ETH/USD"], "2025-01-01", "2025-10-01", "1Hour", asset_class="crypto")
returns = log_returns(prices)
print(len(returns), "hourly returns,", returns.index[0], "to", returns.index[-1])

for x, y in [("BTC/USD", "ETH/USD"), ("ETH/USD", "BTC/USD")]:
    rep = report(to_array(returns, x, y), Config(n_surrogates=100))
    results.append(summarize(rep, f"crypto 1h: {x} -> {y}"))
pd.DataFrame(results[-2:]).set_index("analysis")

### 3b. Stocks, daily, market factor removed

A few hundred daily bars is too few for return-sized effects; this uses ten years.

In [ ]:
X, Y, MARKET = "AAPL", "MSFT", "SPY"

if have_stock_keys():
    prices = fetch_bars([X, Y, MARKET], "2016-01-01", "2025-12-31", "1Day")
    data = to_array(log_returns(prices), x=X, y=Y, refs=[MARKET])
    rep = report(data, Config(ref_cols=(2,), n_surrogates=100))
    results.append(summarize(rep, f"daily: {X} -> {Y} | {MARKET} removed"))
    display(pd.DataFrame(results[-1:]).set_index("analysis"))
else:
    print("Skipped: add APCA_API_KEY_ID and APCA_API_SECRET_KEY as Colab secrets to run this.")

### 3c. Stocks, 1-minute bars, one session per trading day

No window spans the overnight gap, and the most recent 20% of days are held out whole as test.

In [ ]:
if have_stock_keys():
    prices = fetch_bars([X, Y, MARKET], "2025-06-01", "2025-09-30", "1Min")
    sessions = intraday_sessions(prices, x=X, y=Y, refs=[MARKET])
    test_days = last_sessions(sessions, 0.2)
    print(len(sessions), "trading days; test days:", test_days[0], "to", test_days[-1])
    rep = report(sessions, Config(ref_cols=(2,), n_surrogates=50), test_ids=test_days)
    results.append(summarize(rep, f"1min: {X} -> {Y} | {MARKET} removed"))
    display(pd.DataFrame(results[-1:]).set_index("analysis"))
else:
    print("Skipped: needs Alpaca stock keys (see 3).")

## 4. Your own CSV

Upload a CSV with one column per series. Column order in `to_array` is X (source), Y (target),
then any reference columns. Set `USE_RETURNS = False` if the columns are already stationary.

In [ ]:
X_COL, Y_COL, REF_COLS = "x", "y", []   # edit to match your CSV
USE_RETURNS = True

if IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        df = pd.read_csv(next(iter(uploaded))).dropna()
        df = log_returns(df[[X_COL, Y_COL, *REF_COLS]]) if USE_RETURNS else df
        cfg = Config(ref_cols=tuple(range(2, 2 + len(REF_COLS))), n_surrogates=100)
        rep = report(to_array(df, X_COL, Y_COL, REF_COLS), cfg)
        results.append(summarize(rep, f"csv: {X_COL} -> {Y_COL}"))
        display(pd.DataFrame(results[-1:]).set_index("analysis"))
else:
    print("Upload works in Colab only; locally, load your data with pandas and call report().")

## Summary

**How to read the verdicts**
- *directed X->Y transfer supported*: significant and passes every leak check.
- *dependence present, direction unresolved*: significant, but at least one check failed. This
  is what the noise-leak scenarios produce.
- *no evidence*: not significant.

Positive TE is predictive information, not causation, and not a trading signal on its own.

In [ ]:
summary = pd.DataFrame(results).set_index("analysis")
summary

In [ ]:
SAVE_TO_DRIVE = False   # set True to keep the table after the runtime resets

if SAVE_TO_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    path = "/content/drive/MyDrive/learned_te_results.csv"
    summary.to_csv(path)
    print("saved", path)